In [ ]:
#!/usr/bin/env python
# =========================================================
# EFC-VAE training script — simulation study
#
# Trains EFC-VAE (Extreme Factor-Copula Variational Autoencoder) on the
# four simulation regimes (Models I-IV) and evaluates holdout emulation.
# Implements the model of Section 3 of the paper:
#   - low-rank spatial field B_jt          (Eq. method-lowrank)
#   - Student-t factor V_jt                (Eq. method-tfactor)
#   - positive-stable factor M_jt, shared across sites via knot-level
#     random effects and Wendland aggregation (Eq. method-psfactor-agg,
#     method-psfactor; see Supplementary "Knot-level generation and
#     spatial aggregation of the positive-stable factor")
#   - gated combination A_jt = alpha * [(1-g) V + g M]  (Eq. method-factor-term)
#   - training objective L_total            (Eq. main-total-loss)
#
# Training follows the staged schedule of Supplementary Section
# "Loss function and training settings":
#   Stage 1  reconstruction warm-up (factor off)
#   Stage 2  tail-emphasized fine-tuning, best-Q99 checkpoint
#   Stage 3  factor-copula training (backbone frozen, factor params trained)
#   Stage 4  holdout emulation (oracle marginal, IDW residual transfer)
#   Stage 5  repeated emulation for ARE(u), chi(u), CRPS, MSPE envelopes
# =========================================================

import os
import gc
import copy
import time
import argparse
import datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import norm, rankdata
from scipy.spatial.distance import pdist, squareform
from scipy.spatial import cKDTree

# =========================================================
# Paths and run configuration
# =========================================================
parser = argparse.ArgumentParser(description="EFC-VAE training on simulation data")
parser.add_argument("--data_dir", type=str, default="./data/simulation",
                     help="Directory containing model{I,II,III,IV}/ subfolders "
                          "with X_raw_train.csv, X_raw_holdout.csv, "
                          "stations_train.csv, stations_holdout.csv, "
                          "W_train.csv, W_holdout.csv")
parser.add_argument("--out_dir", type=str, default="./runs",
                     help="Directory where results are written")
parser.add_argument("--models", type=str, default="I,II,III,IV",
                     help="Comma-separated subset of regimes to run")
parser.add_argument("--u_mode", type=str, default="weighted",
                     choices=["tail", "full", "weighted"],
                     help="Quantile-grid setting; 'weighted' is the setting "
                          "used in the paper")
parser.add_argument("--n_emul", type=int, default=100,
                     help="Number of repeated emulations for the envelopes "
                          "(Stage 5)")
parser.add_argument("--gpu", type=int, default=0,
                     help="CUDA device index; -1 for CPU")
parser.add_argument("--seed", type=int, default=2026)
args = parser.parse_args()

SIM_BASE = args.data_dir
MODELS = [m.strip() for m in args.models.split(",") if m.strip()]
U_MODE = args.u_mode
N_EMUL = args.n_emul
GLOBAL_SEED = args.seed

_TODAY = datetime.datetime.now().strftime("%y%m%d")
OUT_BASE = os.path.join(args.out_dir, f"efcvae_sim_{U_MODE}_{_TODAY}")

# =========================================================
# Hyperparameters (Supplementary Table "Main hyperparameters and settings")
# =========================================================
HIDDEN_DIM            = 256   # encoder hidden width H
USE_RESIDUAL_ENCODER  = False
ENCODER_DEPTH         = 3
ENCODER_DROPOUT       = 0.05
BATCH_SIZE            = 32

EPOCHS_PRETRAIN = 5000   # Stage 1 + Stage 2 (reconstruction/tail warm-up)
EPOCHS_FACTOR   = 5000   # Stage 3a + Stage 3b (factor-copula training)

LR_BASE = 1e-3

ALPHA_MAX  = 1.0   # upper bound on alpha(s)          (Table: a_max)
GATE_INIT  = 0.3   # initial/target gate value         (Table: g_target)

LAM_DEPENDENCE      = 70.0   # lambda_dep, split 0.7 (chi) / 0.3 (quantile)
LAM_REGULARIZATION  = 1e-3   # lambda_reg, base parameter regularization
LAM_GATE            = 5e-3   # lambda_gate, gate target scale

CHI_N_PAIRS = 1500   # site pairs used in the chi(u) training loss

# =========================================================
# Repeated-emulation settings (Stage 5: ARE(u), chi(u), CRPS, MSPE)
# =========================================================
ENVELOPE_QS  = (0.025, 0.5, 0.975)                       # envelope quantiles
ARE_U_GRID   = np.round(np.arange(0.80, 0.991, 0.01), 4)  # ARE_psi(u) grid
CHI_U_GRID   = np.round(np.linspace(0.80, 0.99, 20), 4)   # chi(u) envelope grid
CHI_ENV_N_PAIRS  = 3000
CHI_ENV_SEED     = 2026
BOXPLOT_QUANTILES = (0.50, 0.75, 0.95, 0.99)
EMUL_SEED_BASE = 7000

# =========================================================
# Quantile grid Q and chi-threshold grid U_chi
# (Eq. main-total-loss; Supplementary "Loss function and training settings")
# =========================================================
if U_MODE == "tail":
    Q_LOSS_LEVELS = (0.95, 0.99)
    _CHI_U_LEVELS = (0.80, 0.85, 0.90, 0.94, 0.97)
    _QM_QWS = ((0.01, 1.0), (0.05, 1.0), (0.90, 1.0),
               (0.95, 1.5), (0.97, 2.0), (0.99, 2.5))
    _S2_TAIL_ON = True
    _Q_LOSS_WEIGHTS = None

elif U_MODE == "full":
    Q_LOSS_LEVELS = (0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80,
                      0.85, 0.90, 0.925, 0.95, 0.975, 0.99)
    _RAW_QW = {0.10: 1.0, 0.20: 1.0, 0.30: 1.0, 0.40: 1.0, 0.50: 1.0,
               0.60: 1.0, 0.70: 1.0, 0.80: 1.0, 0.85: 1.0,
               0.90: 2.0, 0.925: 2.5, 0.95: 3.0, 0.975: 4.0, 0.99: 5.0}
    _w_mean = float(np.mean(list(_RAW_QW.values())))
    _NORM_QW = {q: w / _w_mean for q, w in _RAW_QW.items()}
    _CHI_U_LEVELS = (0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.975)
    _QM_QWS = tuple((q, _NORM_QW[q]) for q in Q_LOSS_LEVELS)
    _S2_TAIL_ON = True
    _Q_LOSS_WEIGHTS = dict(_NORM_QW)

else:  # "weighted" — the setting used in the paper (Q, U_chi as reported)
    Q_LOSS_LEVELS = (0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99)
    _CHI_U_LEVELS = (0.50, 0.70, 0.80, 0.90, 0.94, 0.97)
    # {0.95, 0.99} carry 70% of the total quantile-loss weight (0.4:0.6 split);
    # the remaining five quantiles share the other 30% equally.
    _QM_QWS = ((0.10, 0.42), (0.25, 0.42), (0.50, 0.42), (0.75, 0.42),
               (0.90, 0.42), (0.95, 1.96), (0.99, 2.94))
    _S2_TAIL_ON = True
    _Q_LOSS_WEIGHTS = {0.10: 0.42, 0.25: 0.42, 0.50: 0.42, 0.75: 0.42,
                       0.90: 0.42, 0.95: 1.96, 0.99: 2.94}

# =========================================================
# Internal schedule and loss-weight constants
# (Supplementary "Loss function and training settings")
# =========================================================
_S1_RATIO  = 0.50   # Stage 1 fraction of EPOCHS_PRETRAIN
_S3A_RATIO = 0.50   # Stage 3a fraction of EPOCHS_FACTOR

_LR_S1 = LR_BASE * 0.5
_LR_S2 = LR_BASE * 0.1
_LR_S3 = LR_BASE * 2.0
_LR_FACTOR_NU    = LR_BASE * 10
_LR_FACTOR_THETA = LR_BASE * 8
_LR_FACTOR_APS   = LR_BASE * 5
_LR_FACTOR_GATE  = LR_BASE * 10
_LR_DECAY_FACTOR = 0.5

# lambda_chi = 0.7 * lambda_dep, lambda_Q = 0.3 * lambda_dep
_LAM_CHI = LAM_DEPENDENCE * 0.7
_LAM_Q99 = LAM_DEPENDENCE * 0.3
_LAM_QM  = LAM_DEPENDENCE * 0.03
_LAM_REC = 1.0
_LAM_TW  = 1.0

_LAM_TW_S2   = 3.0
_LAM_QM_S2   = 5.0
_LAM_Q99_S2  = 25.0
_LAM_Q995_S2 = 15.0
_ALPHA_TW_S2 = 2.5

_LAM_ALPHA_REG   = LAM_REGULARIZATION
_LAM_NU_REG      = LAM_REGULARIZATION * 1e-4
_LAM_NU_SMOOTH   = LAM_REGULARIZATION * 0.1     # knot-level smoothness of nu(s)
_LAM_THETA_SMOOTH = LAM_REGULARIZATION          # knot-level smoothness of theta(s)

_LAM_GATE_TARGET = LAM_GATE                     # pulls gate(s) toward g_target
_LAM_GATE_SMOOTH = LAM_GATE * 0.002              # knot-level smoothness of gate(s)
_LAM_GATE_L1     = 0.0                          # gate sparsity penalty (off by default)

_ALPHA_TW = 1.5
_TAU_TW   = 0.95
_BETA_KL  = 0.1   # KL weight beta_KL (Eq. main-total-loss)

# Dependence-component ranges (Section "Spatial parameterization of the
# dependence parameters"; Supplementary hyperparameter table)
_NU_INIT, _NU_MIN, _NU_MAX = 5.0, 2.5, 500.0
_ALPHA_INIT = 0.50
_ALPHA_PS_INIT, _ALPHA_PS_MIN, _ALPHA_PS_MAX = 0.5, 0.1, 0.95
_THETA_INIT, _THETA_MIN, _THETA_MAX = 2.0, 0.05, 20.0

_M0_CLIP_MAX = 6.0
_V_0_CLIP = 5.0
_GATE_THRESHOLD = 0.03   # skip positive-stable sampling if gate is negligible

_CHI_TEMP = 10.0         # soft-threshold temperature for the chi(u) loss
_CHI_WEIGHT_EXP = 1.0

_SPATIAL_INV_NEIGHBORS = 8     # IDW residual transfer (Eq. kmrt-idw)
_SPATIAL_INV_BANDWIDTH = 0.5

_EVAL_EVERY = 10
_BEST_TRACK_EP_MIN = 10
_PRINT_EVERY = 10

N_EPOCHS_S1  = int(EPOCHS_PRETRAIN * _S1_RATIO)
N_EPOCHS_S2  = EPOCHS_PRETRAIN - N_EPOCHS_S1
N_EPOCHS_S3A = int(EPOCHS_FACTOR * _S3A_RATIO)
N_EPOCHS_S3B = EPOCHS_FACTOR - N_EPOCHS_S3A

# =========================================================
# Device
# =========================================================
if args.gpu < 0 or not torch.cuda.is_available():
    device = torch.device("cpu")
else:
    n_gpu = torch.cuda.device_count()
    gpu_id = args.gpu if args.gpu < n_gpu else 0
    device = torch.device(f"cuda:{gpu_id}")
    torch.cuda.set_device(device)

np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"device: {device}")
print(f"data_dir: {SIM_BASE}")
print(f"out_dir : {OUT_BASE}")
print(f"models  : {MODELS}   u_mode: {U_MODE}   n_emul: {N_EMUL}")


# =========================================================
# Marginal transforms (rank-PIT, used for the simulation study;
# Red Sea application uses GEV-PIT instead — see redsea scripts)
# =========================================================
def rank_pit_per_site(X, eps=1e-4):
    """Per-site rank-based PIT: X (n_site, n_rep) -> uniform scores."""
    n_site, n_rep = X.shape
    U = np.zeros_like(X, dtype=np.float32)
    for i in range(n_site):
        r = rankdata(X[i], method="average")
        U[i] = np.clip(r / (n_rep + 1), eps, 1 - eps)
    return U


def to_gaussian_z(U):
    """Standard-normal inverse CDF, clipped to [-6, 6] for stability."""
    Z = norm.ppf(np.clip(U, 1e-4, 1 - 1e-4))
    return np.clip(Z, -6, 6).astype(np.float32)


def empirical_inverse_per_site(U_pred, X_ref):
    """Per-site empirical inverse CDF (spatial-empirical inverse transform,
    Section 'Holdout emulation and the inverse transform')."""
    n_site, _ = U_pred.shape
    X_out = np.zeros_like(U_pred, dtype=np.float32)
    for i in range(n_site):
        ref_sorted = np.sort(X_ref[i])
        n_ref = len(ref_sorted)
        probs = np.arange(1, n_ref + 1) / (n_ref + 1)
        X_out[i] = np.interp(U_pred[i], probs, ref_sorted)
    return X_out


def idw_interpolate_matrix(M_ref, stations_pred, stations_ref,
                            n_neighbors=_SPATIAL_INV_NEIGHBORS,
                            bandwidth=_SPATIAL_INV_BANDWIDTH):
    """Coordinate-based inverse-distance-weighted interpolation, used to
    transfer the training residual field R_jt to holdout locations
    (Eq. kmrt-idw)."""
    n_pred = stations_pred.shape[0]
    n_ref = stations_ref.shape[0]
    D = np.sqrt(((stations_pred[:, None, :] - stations_ref[None, :, :]) ** 2).sum(axis=2))
    k = min(n_neighbors, n_ref)
    nearest_idx = np.argpartition(D, k - 1, axis=1)[:, :k]
    nearest_dists = np.take_along_axis(D, nearest_idx, axis=1)
    weights = np.exp(-(nearest_dists ** 2) / (bandwidth ** 2 + 1e-8))
    weights = weights / np.maximum(weights.sum(axis=1, keepdims=True), 1e-12)
    M_out = np.zeros((n_pred, M_ref.shape[1]), dtype=np.float32)
    for i in range(n_pred):
        M_out[i] = (weights[i, :, None] * M_ref[nearest_idx[i]]).sum(axis=0)
    return M_out


def z_to_X_oracle(Z_pred, X_holdout_ref):
    """Map common-scale predictions back to the data scale using the
    standard normal CDF followed by the per-site empirical inverse
    (oracle marginal: the holdout marginal is used only at this step)."""
    U = norm.cdf(np.clip(Z_pred, -6, 6))
    return empirical_inverse_per_site(U, X_holdout_ref)


def maybe_T(arr, n_site):
    if arr.shape[0] == n_site:
        return arr
    if arr.shape[1] == n_site:
        return arr.T
    raise ValueError("shape mismatch between array and station count")


# =========================================================
# Diagnostics: ARE_psi(u) and chi(u)   (Appendix "Definitions of the
# evaluation metrics"; Eqs. are-def, and the chi(u) estimator of
# Section 4.1)
# =========================================================
def empirical_pit_TJ(X_TJ, eps=1e-6):
    X_TJ = np.asarray(X_TJ, dtype=float)
    T = X_TJ.shape[0]
    ranks = np.argsort(np.argsort(X_TJ, axis=0), axis=0) + 1.0
    return np.clip(ranks / (T + 1.0), eps, 1 - eps)


def compute_are_curve_from_X(X_field, coords, u_grid, ref_idx=None, psi=None):
    """ARE_psi(u): averaged radius of exceedances around a reference site
    (Eq. are-def). ref_idx defaults to the site closest to the coordinate
    median; psi defaults to the median nearest-neighbor distance."""
    X_field = np.asarray(X_field, dtype=float)
    coords = np.asarray(coords, dtype=float)
    U = empirical_pit_TJ(X_field.T)
    if ref_idx is None:
        centre = np.median(coords, axis=0)
        ref_idx = int(np.argmin(np.linalg.norm(coords - centre[None, :], axis=1)))
    if psi is None:
        nn_d = cKDTree(coords).query(coords, k=2)[0][:, 1]
        psi = float(np.median(nn_d))
    U_ref = U[:, ref_idx]
    out = np.full(len(u_grid), np.nan, dtype=float)
    for k, u in enumerate(u_grid):
        ref_exc = U_ref > u
        n_ref = int(ref_exc.sum())
        if n_ref < 1:
            continue
        joint = ((U > u) & ref_exc[:, None]).sum()
        out[k] = np.sqrt((psi * psi * joint) / (np.pi * n_ref))
    return out, ref_idx, psi


def build_fixed_chi_pairs(coords, n_pairs=CHI_ENV_N_PAIRS, seed=CHI_ENV_SEED):
    """Fixed, distance-stratified site pairs used for the chi(u) envelope,
    so that the truth and every emulation are evaluated on the same pairs."""
    rng = np.random.default_rng(seed)
    n = coords.shape[0]
    sub = np.arange(n)
    D = squareform(pdist(coords[sub]))
    iu = np.triu_indices(len(sub), k=1)
    di = D[iu]
    ii_sub, jj_sub = np.array(iu[0]), np.array(iu[1])
    dmax = float(di.max())
    edges = np.array([0.00, 0.04, 0.08, 0.13, 0.20, 0.30, 0.45, 0.65, 1.01]) * dmax
    per = max(1, n_pairs // (len(edges) - 1))
    pi, pj = [], []
    for kb in range(len(edges) - 1):
        m = (di >= edges[kb]) & (di < edges[kb + 1])
        nin = int(m.sum())
        if nin == 0:
            continue
        ch = rng.choice(np.where(m)[0], min(per, nin), replace=False)
        pi.append(sub[ii_sub[ch]])
        pj.append(sub[jj_sub[ch]])
    return np.concatenate(pi), np.concatenate(pj)


def chi_u_curve_fixed(X_field, u_grid, ii, jj):
    """chi(u) on a fixed set of site pairs (ii, jj)."""
    X = np.asarray(X_field, dtype=float)
    n_rep = X.shape[1]
    R = np.argsort(np.argsort(X, axis=1), axis=1).astype(float)
    U = (R + 1.0) / (n_rep + 1.0)
    Ui, Uj = U[ii], U[jj]
    out = np.empty(len(u_grid))
    for k, u in enumerate(u_grid):
        a, b = Ui > u, Uj > u
        denom = b.mean(axis=1)
        joint = (a & b).mean(axis=1)
        m = denom > 1e-6
        out[k] = (joint[m] / denom[m]).mean() if m.any() else np.nan
    return out


def site_crps(x_pred_samples, x_truth):
    """Sample-based CRPS at a single (site, block); Eq. crps."""
    xp = np.asarray(x_pred_samples, dtype=float)
    term1 = np.mean(np.abs(xp - x_truth))
    term2 = 0.5 * np.mean(np.abs(xp[:, None] - xp[None, :]))
    return term1 - term2


# =========================================================
# Loss terms (Eq. main-total-loss and Supplementary
# "Loss function and training settings")
# =========================================================
def tail_weighted_mse(z_pred, z_true, alpha=_ALPHA_TW, tau=_TAU_TW):
    z_tau = torch.quantile(z_true, tau)
    w = torch.exp(alpha * F.relu(z_true - z_tau))
    return torch.mean(w * (z_pred - z_true) ** 2)


def quantile_match_loss(z_pred, z_true, qws=_QM_QWS):
    return sum(w * F.mse_loss(torch.quantile(z_pred, q, dim=0),
                               torch.quantile(z_true, q, dim=0))
               for q, w in qws)


def kl_gaussian_loss(mu, lv):
    """KL divergence to N(0, I); Eq. main-kl-loss."""
    return -0.5 * torch.mean(1 + lv - mu ** 2 - lv.exp())


def stage1_loss(z_pred, z_true, mu, lv):
    """Stage 1: reconstruction warm-up, factor off."""
    l_rec = F.mse_loss(z_pred, z_true)
    l_tw = tail_weighted_mse(z_pred, z_true, _ALPHA_TW, _TAU_TW)
    l_qm = quantile_match_loss(z_pred, z_true)
    l_kl = kl_gaussian_loss(mu, lv)
    tw_coef = _LAM_TW if _S2_TAIL_ON else 0.0
    total = l_rec + tw_coef * l_tw + 0.5 * l_qm + _BETA_KL * l_kl
    return total, dict(rec=l_rec.item(), tw=l_tw.item(), qm=l_qm.item(), kl=l_kl.item())


def stage2_loss(z_pred, z_true):
    """Stage 2: tail-emphasized fine-tuning with Q99/Q99.5 targets used
    for best-checkpoint tracking."""
    l_rec = F.mse_loss(z_pred, z_true)
    l_qm = quantile_match_loss(z_pred, z_true)
    if _S2_TAIL_ON:
        l_tw = tail_weighted_mse(z_pred, z_true, _ALPHA_TW_S2, _TAU_TW)
        l_q99 = F.mse_loss(torch.quantile(z_pred, 0.99, dim=0),
                            torch.quantile(z_true, 0.99, dim=0))
        l_q995 = F.mse_loss(torch.quantile(z_pred, 0.995, dim=0),
                             torch.quantile(z_true, 0.995, dim=0))
        total = (l_rec + _LAM_TW_S2 * l_tw + _LAM_QM_S2 * l_qm
                 + _LAM_Q99_S2 * l_q99 + _LAM_Q995_S2 * l_q995)
        return total, dict(rec=l_rec.item(), q99=l_q99.item())
    total = l_rec + _LAM_QM_S2 * l_qm
    with torch.no_grad():
        q99_dbg = F.mse_loss(torch.quantile(z_pred, 0.99, dim=0),
                              torch.quantile(z_true, 0.99, dim=0)).item()
    return total, dict(rec=l_rec.item(), q99=q99_dbg)


_CHI_THRESHOLDS = torch.tensor([float(norm.ppf(u)) for u in _CHI_U_LEVELS],
                                dtype=torch.float32)


def compute_chi_soft(z, pair_i, pair_j, thresholds=_CHI_THRESHOLDS, temp=_CHI_TEMP):
    """Soft-threshold approximation to the empirical chi(u), used inside
    the training loop (Section 'Loss function and training settings')."""
    z_i, z_j = z[:, pair_i], z[:, pair_j]
    th = thresholds.to(z.device)
    chis = []
    for ui in range(th.shape[0]):
        a = torch.sigmoid((z_i - th[ui]) * temp)
        b = torch.sigmoid((z_j - th[ui]) * temp)
        chi = (a * b).mean(dim=0) / b.mean(dim=0).clamp(min=1e-6)
        chis.append(chi)
    return torch.stack(chis, dim=0)


def estimate_gate_target(chi_target):
    """Data-driven gate target g_target, derived from the empirical
    chi(u) at the highest threshold (bounded to [0.15, 0.65])."""
    chi_high = chi_target[-1].mean().item()
    return torch.tensor(max(0.15, min(0.65, chi_high * 1.6)), dtype=torch.float32)


def stage3_loss(z_recon, z_input, z_gen, chi_target, alpha_basis,
                 log_nu_basis, log_theta_basis, gate_basis, gate_per_site,
                 gate_target, pair_i, pair_j):
    """Stage 3: factor-copula training. Combines reconstruction,
    tail-dependence (chi), quantile-calibration, and regularization terms
    (Eq. main-total-loss)."""
    l_rec = F.mse_loss(z_recon, z_input)
    l_qm = quantile_match_loss(z_recon, z_input)

    chi_gen = compute_chi_soft(z_gen, pair_i, pair_j)
    chi_weights = (chi_target.detach() ** _CHI_WEIGHT_EXP + 0.05)
    chi_weights = chi_weights / chi_weights.mean()
    l_chi = (chi_weights * (chi_gen - chi_target) ** 2).mean()

    if U_MODE == "tail":
        q99_p = torch.quantile(z_gen, 0.99, dim=0)
        q99_t = torch.tensor(2.326, device=z_gen.device)  # standard-normal Q99
        l_q = ((q99_p - q99_t) ** 2).mean()
    else:
        l_q, w_sum = 0.0, 0.0
        for q in Q_LOSS_LEVELS:
            w = _Q_LOSS_WEIGHTS.get(q, 1.0) if _Q_LOSS_WEIGHTS else 1.0
            qp = torch.quantile(z_gen, q, dim=0)
            qt = torch.tensor(float(norm.ppf(q)), device=z_gen.device)
            l_q = l_q + w * ((qp - qt) ** 2).mean()
            w_sum += w
        l_q = l_q / max(w_sum, 1e-8)

    # Parameter regularization and knot-level smoothness
    # (Section 'Spatial parameterization of the dependence parameters')
    l_alpha_reg = (alpha_basis ** 2).mean()
    nu_vals = _NU_MIN + torch.exp(log_nu_basis)
    l_nu_smooth = nu_vals.var()
    l_nu_reg = (1.0 / (nu_vals + 1.0)).mean()  # discourages nu collapsing to nu_min
    theta_vals = _THETA_MIN + torch.exp(log_theta_basis)
    l_theta_smooth = theta_vals.var()
    l_gate_target = ((gate_per_site - gate_target) ** 2).mean()
    l_gate_smooth = gate_basis.var()
    # L1 sparsity on the gate: regimes where chi_target favors a nonzero
    # gate (e.g. asymptotic dependence) will retain it against this
    # penalty; regimes with little benefit will shrink it toward zero.
    l_gate_l1 = gate_basis.abs().mean()

    total = (_LAM_REC * l_rec + _LAM_CHI * l_chi + _LAM_QM * l_qm
             + _LAM_Q99 * l_q + _LAM_ALPHA_REG * l_alpha_reg
             + _LAM_NU_REG * l_nu_reg + _LAM_NU_SMOOTH * l_nu_smooth
             + _LAM_THETA_SMOOTH * l_theta_smooth
             + _LAM_GATE_TARGET * l_gate_target + _LAM_GATE_SMOOTH * l_gate_smooth
             + _LAM_GATE_L1 * l_gate_l1)

    q_log = l_q.item() if torch.is_tensor(l_q) else float(l_q)
    return total, dict(rec=l_rec.item(), chi=l_chi.item(), q99=q_log,
                        gate_target=l_gate_target.item(), gate_l1=l_gate_l1.item())


# =========================================================
# Model (Section 3, "A factor-copula variational autoencoder")
# =========================================================
class ResidualMLPBlock(nn.Module):
    """Optional residual MLP block for a deeper encoder (not used in the
    reported results; USE_RESIDUAL_ENCODER = False)."""

    def __init__(self, dim, dropout=ENCODER_DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.LayerNorm(dim), nn.SiLU(),
            nn.Dropout(dropout), nn.Linear(dim, dim), nn.LayerNorm(dim))
        self.act = nn.SiLU()

    def forward(self, x):
        return self.act(x + self.net(x))


class ResidualEncoder(nn.Module):
    def __init__(self, n_site, hidden=HIDDEN_DIM, depth=ENCODER_DEPTH,
                 dropout=ENCODER_DROPOUT):
        super().__init__()
        layers = [nn.Linear(n_site, hidden), nn.LayerNorm(hidden), nn.SiLU(),
                  nn.Dropout(dropout)]
        layers += [ResidualMLPBlock(hidden, dropout=dropout) for _ in range(depth)]
        layers += [nn.Linear(hidden, 128), nn.LayerNorm(128), nn.SiLU()]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class CopulaVAE(nn.Module):
    """EFC-VAE. Encoder q_phi(c_t | Z_t) (Eq. method-vae-q); decoder
    combines the low-rank field B_jt, factor-copula term A_jt, and the
    site--block residual b_jt (Eq. main-decoder, conditional mean)."""

    def __init__(self, n_site, n_basis, n_rep, hidden=HIDDEN_DIM, factor=False):
        super().__init__()
        self.n_site, self.n_basis, self.n_rep = n_site, n_basis, n_rep
        self.factor = factor
        # latent dim = n_basis (basis coefficients beta_t) + 1 (common
        # Gaussian factor H_t) when the factor-copula term is active
        out_dim = n_basis + (1 if factor else 0)
        self.latent_dim = out_dim

        if USE_RESIDUAL_ENCODER:
            self.enc = ResidualEncoder(n_site, hidden=hidden)
        else:
            self.enc = nn.Sequential(
                nn.Linear(n_site, hidden), nn.LayerNorm(hidden), nn.SiLU(),
                nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.SiLU(),
                nn.Linear(hidden, 128), nn.LayerNorm(128), nn.SiLU())
        self.fc_mu = nn.Linear(128, out_dim)
        self.fc_lv = nn.Linear(128, out_dim)

        # site--block residual field R_jt (per-site free parameter)
        self.b = nn.Parameter(torch.zeros(n_site, n_rep))

        if factor:
            # Knot-level raw parameters rho^(p), p in {alpha, nu, theta, gate};
            # mapped to their valid ranges by the link functions below
            # (Section 'Spatial parameterization of the dependence parameters').
            self.log_alpha_basis = nn.Parameter(
                torch.log(torch.full((n_basis,), float(_ALPHA_INIT))))
            self.log_nu_basis = nn.Parameter(
                torch.full((n_basis,), float(np.log(_NU_INIT - _NU_MIN + 1e-6))))
            self.logit_alpha_ps = nn.Parameter(
                torch.tensor(float(np.log(
                    (_ALPHA_PS_INIT - _ALPHA_PS_MIN) / (_ALPHA_PS_MAX - _ALPHA_PS_INIT)))))
            self.log_theta_basis = nn.Parameter(
                torch.full((n_basis,), float(np.log(_THETA_INIT - _THETA_MIN + 1e-6))))
            init_logit_g = float(np.log(GATE_INIT / (1 - GATE_INIT)))
            self.logit_gate_basis = nn.Parameter(
                torch.full((n_basis,), init_logit_g) + torch.randn(n_basis) * 0.2)

    # ---- knot-level parameters mapped to their valid ranges, then
    #      interpolated to site level with the row-normalized Wendland
    #      basis W (Eq. smooth-params) ----
    def alpha_basis(self):
        return torch.clamp(torch.exp(self.log_alpha_basis), 1e-4, ALPHA_MAX)

    def alpha_per_site(self, W):
        return W @ self.alpha_basis()

    def nu_basis(self):
        return torch.clamp(_NU_MIN + torch.exp(self.log_nu_basis), _NU_MIN, _NU_MAX)

    def nu_per_site(self, W):
        return W @ self.nu_basis()

    def alpha_ps(self):
        raw = torch.sigmoid(self.logit_alpha_ps)
        return _ALPHA_PS_MIN + (_ALPHA_PS_MAX - _ALPHA_PS_MIN) * raw

    def theta_basis(self):
        return torch.clamp(_THETA_MIN + torch.exp(self.log_theta_basis),
                            _THETA_MIN, _THETA_MAX)

    def theta_per_site(self, W):
        return W @ self.theta_basis()

    def gate_basis_val(self):
        return torch.sigmoid(self.logit_gate_basis)

    def gate_per_site(self, W):
        return W @ self.gate_basis_val()

    def encode(self, x):
        h = self.enc(x)
        mu = self.fc_mu(h)
        lv = torch.clamp(self.fc_lv(h), -4, 4)
        return mu, lv

    def reparam(self, mu, lv, det=False):
        if det:
            return mu
        return mu + torch.randn_like(mu) * torch.exp(0.5 * lv)

    def sample_V_0(self, h_gauss, W):
        """Student-t factor V_jt = H_t / sqrt(Q_jt / nu_j), Q_jt ~ chi2_nu_j
        (Eq. method-tfactor)."""
        nu_site = self.nu_per_site(W)
        BS, n_site = h_gauss.shape[0], W.shape[0]
        nu_e = nu_site.unsqueeze(0).expand(BS, -1)
        chi2 = torch.distributions.Gamma(
            concentration=nu_e / 2.0,
            rate=torch.tensor(0.5, device=h_gauss.device)).rsample()
        chi2 = torch.clamp(chi2, min=1e-4)
        h_e = h_gauss.unsqueeze(1).expand(-1, n_site)
        v_t = h_e / torch.sqrt(chi2 / nu_e)
        return torch.clamp(v_t, -_V_0_CLIP * 1.5, _V_0_CLIP * 1.5)

    def sample_M_0_shared(self, BS, theta_per_site_val, W):
        """Positive-stable factor, generated at the knot level and
        aggregated with the Wendland weights (Supplementary 'Knot-level
        generation and spatial aggregation of the positive-stable
        factor'):

            Lambda_k ~ PS(alpha_PS)                      (Chambers-Mallows-Stuck)
            S_j      = ( sum_k W_jk^(1/alpha_PS) Lambda_k )^alpha_PS
            M_j      = S_j * exp(-theta_j * S_j^alpha_PS)

        Computed in log-sum-exp form with the site dimension chunked, so
        that large grids (e.g. 16,703 Red Sea sites) fit in memory.
        """
        alpha = self.alpha_ps()
        device = W.device
        n_site, n_knot = W.shape

        # CMS sampling of Lambda_k ~ PS(alpha), shared across sites
        u = (torch.rand(BS, n_knot, device=device) * np.pi).clamp(1e-4, np.pi - 1e-4)
        e = -torch.log(torch.rand(BS, n_knot, device=device).clamp(1e-6))
        log_z = (torch.log(torch.sin(alpha * u).clamp(min=1e-8))
                  + ((1 - alpha) / alpha) * (
                      torch.log(torch.sin((1 - alpha) * u).clamp(min=1e-8))
                      - torch.log(torch.sin(u).clamp(min=1e-8))))
        log_lambda = torch.clamp(
            log_z - ((1 - alpha) / alpha) * torch.log(e.clamp(min=1e-6)),
            min=-10, max=8)  # (BS, n_knot)

        logW = torch.log(W.clamp(min=1e-12))  # (n_site, n_knot)
        theta = theta_per_site_val
        site_chunk = 4000
        out = torch.empty(BS, n_site, device=device)
        for s0 in range(0, n_site, site_chunk):
            s1 = min(s0 + site_chunk, n_site)
            term = (logW[s0:s1].unsqueeze(0) / alpha) + log_lambda.unsqueeze(1)
            log_S = torch.clamp(alpha * torch.logsumexp(term, dim=2), min=-10, max=8)
            S_pow_alpha = torch.exp(alpha * log_S)
            log_M = torch.clamp(log_S - theta[s0:s1].unsqueeze(0) * S_pow_alpha,
                                 min=-8, max=_M0_CLIP_MAX)
            M = torch.exp(log_M)
            # Log-ratio to the site-specific mean, then scale/clip to put
            # M on the same standard-normal scale as the other terms.
            M_mean = M.mean(dim=0, keepdim=True) + 1e-6
            out[:, s0:s1] = torch.clamp(torch.log(M / M_mean + 0.1) * 0.5, -3.0, 3.0)
        return out

    def decode(self, z, W, t_idx=None):
        """Returns the conditional mean B_jt + A_jt (+ R_jt if t_idx is
        given); the observation-noise term eta_jt is not added explicitly
        and enters only through the reconstruction loss (Eq. main-decoder)."""
        if self.factor:
            c = z[:, :self.n_basis]
            h_gauss = z[:, self.n_basis]
            out = c @ W.T  # B_jt = sum_k W_jk beta_kt

            V_0 = self.sample_V_0(h_gauss, W)
            g = self.gate_per_site(W)
            if g.max().item() > _GATE_THRESHOLD:
                M_0 = self.sample_M_0_shared(z.shape[0], self.theta_per_site(W), W)
            else:
                M_0 = torch.zeros_like(V_0)

            F_combined = (1 - g).unsqueeze(0) * V_0 + g.unsqueeze(0) * M_0
            alpha = self.alpha_per_site(W)
            out = out + alpha.unsqueeze(0) * F_combined  # A_jt (Eq. method-factor-term)
        else:
            out = z @ W.T

        if t_idx is not None and self.b.shape[0] == out.shape[1]:
            out = out + self.b[:, t_idx].T  # + R_jt (training sites only)
        return out

    def forward(self, x, W, t_idx=None, det=False):
        mu, lv = self.encode(x)
        z = self.reparam(mu, lv, det=det)
        return self.decode(z, W, t_idx=t_idx), mu, lv, z


# =========================================================
# Stage 5: repeated emulation for the ARE(u), chi(u), CRPS, MSPE envelopes
# =========================================================
def run_multi_emulation(model_efc, Z_tr_T_t, W_va_t, b_va_interp, X_va, sta_va,
                         out_dir, model_name, n_emul=N_EMUL):
    if n_emul is None or n_emul < 1:
        print("  [multi-emul] n_emul < 1, skipping")
        return None

    print(f"  Repeated emulation (n_emul={n_emul}) for box/ARE/chi/CRPS envelopes")
    model_efc.eval()
    n_va_site, n_rep = X_va.shape
    coords_va = sta_va

    q_levels = list(BOXPLOT_QUANTILES)
    q_truth = {q: np.quantile(X_va, q, axis=1) for q in q_levels}

    are_truth, ref_idx, psi = compute_are_curve_from_X(X_va, coords_va, ARE_U_GRID)
    n_boot = min(n_emul, 100)
    rng_boot = np.random.default_rng(EMUL_SEED_BASE + 99999)
    are_truth_boot = np.full((n_boot, len(ARE_U_GRID)), np.nan)
    for bi in range(n_boot):
        cols = rng_boot.integers(0, n_rep, size=n_rep)
        are_truth_boot[bi], _, _ = compute_are_curve_from_X(
            X_va[:, cols], coords_va, ARE_U_GRID, ref_idx=ref_idx, psi=psi)

    chi_ii, chi_jj = build_fixed_chi_pairs(coords_va, CHI_ENV_N_PAIRS, CHI_ENV_SEED)
    chi_truth_single = chi_u_curve_fixed(X_va, CHI_U_GRID, chi_ii, chi_jj)
    chi_truth_boot = np.full((n_boot, len(CHI_U_GRID)), np.nan)
    for bi in range(n_boot):
        cols = rng_boot.integers(0, n_rep, size=n_rep)
        chi_truth_boot[bi] = chi_u_curve_fixed(X_va[:, cols], CHI_U_GRID, chi_ii, chi_jj)

    are_stack = np.full((n_emul, len(ARE_U_GRID)), np.nan)
    chi_stack = np.full((n_emul, len(CHI_U_GRID)), np.nan)
    qcurve_stack = np.full((n_emul, len(ARE_U_GRID)), np.nan)
    qcurve_truth = np.array([np.quantile(X_va, u) for u in ARE_U_GRID])
    site_metric_rows = []
    n_raw_save = min(5, n_emul)
    tag = f"_{U_MODE}"

    t0 = time.time()
    for r in range(n_emul):
        torch.manual_seed(EMUL_SEED_BASE + r)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(EMUL_SEED_BASE + r)
        with torch.no_grad():
            z_out, _, _, _ = model_efc(Z_tr_T_t, W_va_t, t_idx=None, det=False)
            Z_pred = z_out.cpu().numpy().T.astype(np.float32)
        Z_pred = (Z_pred + b_va_interp).astype(np.float32)
        X_gen = z_to_X_oracle(Z_pred, X_va)

        are_stack[r], _, _ = compute_are_curve_from_X(
            X_gen, coords_va, ARE_U_GRID, ref_idx=ref_idx, psi=psi)
        chi_stack[r] = chi_u_curve_fixed(X_gen, CHI_U_GRID, chi_ii, chi_jj)
        qcurve_stack[r] = np.array([np.quantile(X_gen, u) for u in ARE_U_GRID])

        for q in q_levels:
            err = np.quantile(X_gen, q, axis=1) - q_truth[q]
            for s in range(n_va_site):
                site_metric_rows.append({"emul": r, "site": s,
                                          "metric": f"Q{int(q * 100)}_err",
                                          "value": float(err[s])})
        field_rmse_site = np.sqrt(np.mean((X_gen - X_va) ** 2, axis=1))
        for s in range(n_va_site):
            site_metric_rows.append({"emul": r, "site": s, "metric": "field_rmse",
                                      "value": float(field_rmse_site[s])})

        if r < n_raw_save:
            pd.DataFrame(X_gen).to_csv(
                os.path.join(out_dir, f"holdout_emul{r:03d}{tag}.csv"), index=False)
        if (r + 1) % 20 == 0 or r == n_emul - 1:
            print(f"    emulation {r + 1}/{n_emul}  ({time.time() - t0:.1f}s)")

    # CRPS / MSPE ensemble
    print(f"  CRPS/MSPE ensemble ({n_emul} regenerations)")
    ens = np.full((n_emul, n_va_site, n_rep), np.nan, dtype=np.float32)
    for r in range(n_emul):
        torch.manual_seed(EMUL_SEED_BASE + r)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(EMUL_SEED_BASE + r)
        with torch.no_grad():
            z_out, _, _, _ = model_efc(Z_tr_T_t, W_va_t, t_idx=None, det=False)
            Z_pred = z_out.cpu().numpy().T.astype(np.float32)
        ens[r] = z_to_X_oracle((Z_pred + b_va_interp).astype(np.float32), X_va)

    crps_site = np.zeros(n_va_site)
    mspe_site = np.zeros(n_va_site)
    for s in range(n_va_site):
        crps_t = np.array([site_crps(ens[:, s, t_], X_va[s, t_]) for t_ in range(n_rep)])
        crps_site[s] = crps_t.mean()
        mspe_site[s] = np.mean((ens[:, s, :].mean(axis=0) - X_va[s]) ** 2)
    for s in range(n_va_site):
        site_metric_rows.append({"emul": -1, "site": s, "metric": "CRPS",
                                  "value": float(crps_site[s])})
        site_metric_rows.append({"emul": -1, "site": s, "metric": "MSPE",
                                  "value": float(mspe_site[s])})

    # ---- write outputs ----
    pd.DataFrame(site_metric_rows).assign(model=model_name, u_mode=U_MODE).to_csv(
        os.path.join(out_dir, f"site_metrics_boxplot{tag}.csv"), index=False)

    are_lo, are_md, are_hi = np.nanquantile(are_stack, ENVELOPE_QS, axis=0)
    are_tb_lo, _, are_tb_hi = np.nanquantile(are_truth_boot, ENVELOPE_QS, axis=0)
    pd.DataFrame({"u": ARE_U_GRID, "model": model_name, "u_mode": U_MODE,
                  "are_emul_lo": are_lo, "are_emul_med": are_md, "are_emul_hi": are_hi,
                  "are_truth_single": are_truth,
                  "are_truth_boot_lo": are_tb_lo, "are_truth_boot_hi": are_tb_hi}
                 ).to_csv(os.path.join(out_dir, f"are_envelope{tag}.csv"), index=False)

    chi_lo, chi_md, chi_hi = np.nanquantile(chi_stack, ENVELOPE_QS, axis=0)
    chi_tb_lo, _, chi_tb_hi = np.nanquantile(chi_truth_boot, ENVELOPE_QS, axis=0)
    pd.DataFrame({"u": CHI_U_GRID, "model": model_name, "u_mode": U_MODE,
                  "chi_emul_lo": chi_lo, "chi_emul_med": chi_md, "chi_emul_hi": chi_hi,
                  "chi_truth_single": chi_truth_single,
                  "chi_truth_boot_lo": chi_tb_lo, "chi_truth_boot_hi": chi_tb_hi}
                 ).to_csv(os.path.join(out_dir, f"chi_envelope{tag}.csv"), index=False)

    q_lo, q_md, q_hi = np.nanquantile(qcurve_stack, ENVELOPE_QS, axis=0)
    pd.DataFrame({"u": ARE_U_GRID, "model": model_name, "u_mode": U_MODE,
                  "q_emul_lo": q_lo, "q_emul_med": q_md, "q_emul_hi": q_hi,
                  "q_truth": qcurve_truth}
                 ).to_csv(os.path.join(out_dir, f"quantile_envelope{tag}.csv"), index=False)

    del ens
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {"crps_mean": float(crps_site.mean()), "mspe_mean": float(mspe_site.mean()),
            "ref_idx": ref_idx, "psi": psi, "n_emul": n_emul,
            "chi_emul_med": chi_md, "chi_truth_single": chi_truth_single}


# =========================================================
# Per-regime training and evaluation
# =========================================================
def run_one_model(data_dir, out_dir, model_name):
    print(f"\n{'=' * 60}\nModel {model_name}  (u_mode={U_MODE})\n{'=' * 60}")
    os.makedirs(out_dir, exist_ok=True)

    def load_csv(name):
        path = os.path.join(data_dir, name)
        if not os.path.exists(path):
            raise FileNotFoundError(path)
        return pd.read_csv(path).values.astype(np.float32)

    X_tr = load_csv("X_raw_train.csv")
    X_va = load_csv("X_raw_holdout.csv")
    sta_tr = load_csv("stations_train.csv")
    sta_va = load_csv("stations_holdout.csv")
    W_tr = load_csv("W_train.csv")
    W_va = load_csv("W_holdout.csv")
    if sta_tr.shape[1] >= 3:
        sta_tr = sta_tr[:, -2:]
    if sta_va.shape[1] >= 3:
        sta_va = sta_va[:, -2:]

    n_tr_site, n_va_site = sta_tr.shape[0], sta_va.shape[0]
    X_tr, X_va = maybe_T(X_tr, n_tr_site), maybe_T(X_va, n_va_site)
    W_tr, W_va = maybe_T(W_tr, n_tr_site), maybe_T(W_va, n_va_site)
    # row-normalize the Wendland basis: W_jk (Eq. method-wendland)
    W_tr = W_tr / np.maximum(W_tr.sum(axis=1, keepdims=True), 1e-8)
    W_va = W_va / np.maximum(W_va.sum(axis=1, keepdims=True), 1e-8)
    X_tr = np.clip(X_tr, 1e-6, None).astype(np.float32)
    X_va = np.clip(X_va, 1e-6, None).astype(np.float32)
    n_tr, n_rep = X_tr.shape
    n_basis = W_tr.shape[1]
    print(f"  X_train {X_tr.shape}  X_holdout {X_va.shape}  n_basis={n_basis}")

    # rank-PIT -> standard normal scale (simulation study)
    Z_tr = to_gaussian_z(rank_pit_per_site(X_tr))
    Z_tr_T_t = torch.tensor(Z_tr.T.copy(), dtype=torch.float32, device=device)
    W_tr_t = torch.tensor(W_tr, dtype=torch.float32, device=device)
    W_va_t = torch.tensor(W_va, dtype=torch.float32, device=device)

    # fixed pairs used inside the training loss (short-range subset)
    rng = np.random.default_rng(GLOBAL_SEED)
    D_tr = squareform(pdist(sta_tr))
    iu = np.triu_indices(n_tr_site, k=1)
    pair_dists = D_tr[iu]
    short_mask = pair_dists <= np.median(pair_dists)
    pi, pj = np.array(iu[0])[short_mask], np.array(iu[1])[short_mask]
    if len(pi) > CHI_N_PAIRS:
        idx = rng.choice(len(pi), CHI_N_PAIRS, replace=False)
        pi, pj = pi[idx], pj[idx]
    pair_i_t = torch.tensor(pi, dtype=torch.long, device=device)
    pair_j_t = torch.tensor(pj, dtype=torch.long, device=device)
    BS = min(BATCH_SIZE, n_rep)

    # ---- Stage 1: reconstruction warm-up ----
    model_pre = CopulaVAE(n_tr_site, n_basis, n_rep, factor=False).to(device)
    opt = torch.optim.AdamW(model_pre.parameters(), lr=_LR_S1, weight_decay=1e-4)
    print(f"  Stage 1: {N_EPOCHS_S1} epochs")
    for ep in range(N_EPOCHS_S1):
        model_pre.train()
        perm = torch.randperm(n_rep, device=device)
        for st in range(0, n_rep, BS):
            ids = perm[st:st + BS]
            zb = Z_tr_T_t[ids]
            z_out, mu, lv, _ = model_pre(zb, W_tr_t, t_idx=ids, det=False)
            loss, _ = stage1_loss(z_out, zb, mu, lv)
            if not torch.isfinite(loss):
                continue
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_pre.parameters(), 1.0)
            opt.step()

    # ---- Stage 2: tail-emphasized fine-tuning, best-Q99 checkpoint ----
    for g in opt.param_groups:
        g["lr"] = _LR_S2
    print(f"  Stage 2: {N_EPOCHS_S2} epochs")
    best_q99, best_state = float("inf"), None
    for ep in range(N_EPOCHS_S2):
        model_pre.train()
        ep_q99 = []
        perm = torch.randperm(n_rep, device=device)
        for st in range(0, n_rep, BS):
            ids = perm[st:st + BS]
            zb = Z_tr_T_t[ids]
            z_out, _, _, _ = model_pre(zb, W_tr_t, t_idx=ids, det=False)
            loss, parts = stage2_loss(z_out, zb)
            if not torch.isfinite(loss):
                continue
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_pre.parameters(), 1.0)
            opt.step()
            ep_q99.append(parts["q99"])
        avg_q99 = np.mean(ep_q99)
        if avg_q99 < best_q99 and ep > 100:
            best_q99, best_state = avg_q99, copy.deepcopy(model_pre.state_dict())
    if best_state is not None:
        model_pre.load_state_dict(best_state)
        print(f"    restored best Stage-2 checkpoint, Q99 loss={best_q99:.5f}")
    pretrained_state = copy.deepcopy(model_pre.state_dict())

    # ---- Stage 3: factor-copula training ----
    print("  Stage 3: factor-copula training")
    model = CopulaVAE(n_tr_site, n_basis, n_rep, factor=True).to(device)
    h_state = model.state_dict()
    for k, v in pretrained_state.items():
        if k in h_state and h_state[k].shape == v.shape:
            h_state[k] = v.clone()
        elif k in ("fc_mu.weight", "fc_lv.weight"):
            h_state[k][:n_basis, :] = v
        elif k in ("fc_mu.bias", "fc_lv.bias"):
            h_state[k][:n_basis] = v
    model.load_state_dict(h_state)

    # freeze the pretrained backbone; train only the factor parameters
    trainable = ("log_alpha_basis", "log_nu_basis", "log_theta_basis",
                 "logit_alpha_ps", "logit_gate_basis",
                 "fc_mu.weight", "fc_mu.bias", "fc_lv.weight", "fc_lv.bias")
    for name, p in model.named_parameters():
        p.requires_grad_(name in trainable)

    param_groups = {"enc": [], "nu": [], "theta": [], "aps": [], "gate": []}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name == "log_nu_basis":
            param_groups["nu"].append(p)
        elif name == "log_theta_basis":
            param_groups["theta"].append(p)
        elif name == "logit_alpha_ps":
            param_groups["aps"].append(p)
        elif name == "logit_gate_basis":
            param_groups["gate"].append(p)
        else:
            param_groups["enc"].append(p)

    opt_factor = torch.optim.AdamW([
        {"params": param_groups["enc"], "lr": _LR_S3, "weight_decay": 1e-5},
        {"params": param_groups["nu"], "lr": _LR_FACTOR_NU, "weight_decay": 0},
        {"params": param_groups["theta"], "lr": _LR_FACTOR_THETA, "weight_decay": 0},
        {"params": param_groups["aps"], "lr": _LR_FACTOR_APS, "weight_decay": 0},
        {"params": param_groups["gate"], "lr": _LR_FACTOR_GATE, "weight_decay": 0},
    ])

    model.eval()
    with torch.no_grad():
        chi_target = compute_chi_soft(Z_tr_T_t, pair_i_t, pair_j_t)
        gate_target = estimate_gate_target(chi_target).to(device)
    print(f"    empirical chi(top u) = {chi_target[-1].mean().item():.3f}, "
          f"gate target = {gate_target.item():.3f}")

    best_train_score, best_factor_state, best_ep = float("inf"), None, 0

    def run_stage3(n_ep, ep_offset, tag_name):
        nonlocal best_train_score, best_factor_state, best_ep
        for ep in range(n_ep):
            if tag_name == "S3b" and ep == n_ep // 2:
                for g in opt_factor.param_groups:
                    g["lr"] *= _LR_DECAY_FACTOR
            model.train()
            ep_scores = []
            perm = torch.randperm(n_rep, device=device)
            for st in range(0, n_rep, BS):
                ids = perm[st:st + BS]
                zb = Z_tr_T_t[ids]
                z_recon, _, _, _ = model(zb, W_tr_t, t_idx=ids, det=False)
                n_gen = max(BS * 4, 128)
                z_prior = torch.randn(n_gen, model.latent_dim, device=device)
                z_gen = model.decode(z_prior, W_tr_t, t_idx=None)
                loss, _ = stage3_loss(
                    z_recon, zb, z_gen, chi_target, model.alpha_basis(),
                    model.log_nu_basis, model.log_theta_basis,
                    model.gate_basis_val(), model.gate_per_site(W_tr_t),
                    gate_target, pair_i_t, pair_j_t)
                if not torch.isfinite(loss):
                    continue
                opt_factor.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0)
                opt_factor.step()
                ep_scores.append(float(loss.detach().cpu().item()))

            ep_global = ep_offset + ep
            if ep % _EVAL_EVERY == 0 and ep > _BEST_TRACK_EP_MIN:
                ts = float(np.mean(ep_scores)) if ep_scores else float("inf")
                if ts < best_train_score:
                    best_train_score = ts
                    best_factor_state = copy.deepcopy(model.state_dict())
                    best_ep = ep_global
            if ep % 50 == 0:
                a = model.alpha_per_site(W_tr_t).detach().cpu().numpy()
                gate = model.gate_per_site(W_tr_t).detach().cpu().numpy()
                print(f"    [{tag_name}] epoch {ep:5d}  "
                      f"alpha={a.mean():.2f}  gate={gate.mean():.3f}  "
                      f"best_loss={best_train_score:.4f}@{best_ep}")

    print(f"  Stage 3a: {N_EPOCHS_S3A} epochs")
    run_stage3(N_EPOCHS_S3A, 0, "S3a")
    print(f"  Stage 3b: {N_EPOCHS_S3B} epochs")
    for g in opt_factor.param_groups:
        g["lr"] *= _LR_DECAY_FACTOR
    run_stage3(N_EPOCHS_S3B, N_EPOCHS_S3A, "S3b")
    if best_factor_state is not None:
        model.load_state_dict(best_factor_state)
        print(f"    restored best Stage-3 checkpoint @epoch {best_ep}")

    # ---- Stage 4: holdout emulation (point estimate) ----
    print("  Stage 4: holdout emulation")
    model.eval()
    with torch.no_grad():
        z_out, _, _, _ = model(Z_tr_T_t, W_va_t, t_idx=None, det=True)
        Z_pred_no_res = z_out.cpu().numpy().T.astype(np.float32)
        b_train = model.b.detach().cpu().numpy().astype(np.float32)
    b_va_interp = idw_interpolate_matrix(b_train, sta_va, sta_tr)  # Eq. kmrt-idw
    Z_pred_with_res = (Z_pred_no_res + b_va_interp).astype(np.float32)
    X_no_res = z_to_X_oracle(Z_pred_no_res, X_va)
    X_with_res = z_to_X_oracle(Z_pred_with_res, X_va)  # main result (Eq. kmrt-main)

    def calc_metrics(X_pred, setting_name):
        q = lambda a, p: np.quantile(a, p, axis=1)
        return {
            "setting": setting_name, "u_mode": U_MODE,
            "Q50_rmse": float(np.sqrt(np.mean((q(X_va, 0.50) - q(X_pred, 0.50)) ** 2))),
            "Q75_rmse": float(np.sqrt(np.mean((q(X_va, 0.75) - q(X_pred, 0.75)) ** 2))),
            "Q95_rmse": float(np.sqrt(np.mean((q(X_va, 0.95) - q(X_pred, 0.95)) ** 2))),
            "Q95_corr": float(np.corrcoef(q(X_va, 0.95), q(X_pred, 0.95))[0, 1]),
            "Q99_rmse": float(np.sqrt(np.mean((q(X_va, 0.99) - q(X_pred, 0.99)) ** 2))),
            "Q99_corr": float(np.corrcoef(q(X_va, 0.99), q(X_pred, 0.99))[0, 1]),
            "field_rmse": float(np.sqrt(np.mean((X_va - X_pred) ** 2))),
        }

    met_no_res = calc_metrics(X_no_res, "no_residual")
    met_with_res = calc_metrics(X_with_res, "with_residual")

    tag = f"_{U_MODE}"
    pd.DataFrame(X_no_res).to_csv(os.path.join(out_dir, f"holdout_no_residual{tag}.csv"),
                                    index=False)
    pd.DataFrame(X_with_res).to_csv(os.path.join(out_dir, f"holdout_with_residual{tag}.csv"),
                                      index=False)
    pd.DataFrame([met_no_res, met_with_res]).to_csv(
        os.path.join(out_dir, f"holdout_summary{tag}.csv"), index=False)

    alpha_train = model.alpha_per_site(W_tr_t).detach().cpu().numpy()
    nu_train = model.nu_per_site(W_tr_t).detach().cpu().numpy()
    theta_train = model.theta_per_site(W_tr_t).detach().cpu().numpy()
    gate_train = model.gate_per_site(W_tr_t).detach().cpu().numpy()
    alpha_ps_final = model.alpha_ps().item()

    print(f"  Model {model_name} results:")
    for met in (met_no_res, met_with_res):
        print(f"    [{met['setting']:<14}] Q95={met['Q95_rmse']:.3f} "
              f"Q99={met['Q99_rmse']:.3f} field={met['field_rmse']:.3f}")
    print(f"    alpha={alpha_train.mean():.3f}  nu={nu_train.mean():.2f}  "
          f"gate={gate_train.mean():.3f}  theta={theta_train.mean():.3f}  "
          f"alpha_PS={alpha_ps_final:.3f}")

    # ---- Stage 5: repeated emulation for the envelopes ----
    emul_info = run_multi_emulation(model, Z_tr_T_t, W_va_t, b_va_interp, X_va, sta_va,
                                     out_dir, model_name, n_emul=N_EMUL)

    del model_pre, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    result = {
        "model": model_name, "u_mode": U_MODE,
        "rmse_q95": met_with_res["Q95_rmse"], "rmse_q99": met_with_res["Q99_rmse"],
        "corr_q95": met_with_res["Q95_corr"], "corr_q99": met_with_res["Q99_corr"],
        "field_rmse": met_with_res["field_rmse"],
        "gate_mean": gate_train.mean(), "alpha_mean": alpha_train.mean(),
        "nu_mean": nu_train.mean(), "theta_mean": theta_train.mean(),
        "alpha_ps": alpha_ps_final,
    }
    if emul_info is not None:
        result["crps_mean"] = emul_info["crps_mean"]
        result["mspe_mean"] = emul_info["mspe_mean"]
        result["chi_top_emul"] = float(emul_info["chi_emul_med"][-1])
        result["chi_top_truth"] = float(emul_info["chi_truth_single"][-1])
    return result


# =========================================================
# Main
# =========================================================
if __name__ == "__main__":
    overall_t0 = time.time()
    results = []
    os.makedirs(OUT_BASE, exist_ok=True)

    for mname in MODELS:
        data_dir = os.path.join(SIM_BASE, f"model{mname}")
        out_dir = os.path.join(OUT_BASE, f"model{mname}")
        if not os.path.isdir(data_dir):
            print(f"WARNING: {data_dir} not found, skipping")
            continue
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        t0 = time.time()
        r = run_one_model(data_dir, out_dir, mname)
        r["time_min"] = (time.time() - t0) / 60
        results.append(r)
        print(f"  Model {mname} runtime: {r['time_min']:.1f} min")

    print(f"\n{'=' * 60}\nAll models done "
          f"({(time.time() - overall_t0) / 60:.1f} min)\n{'=' * 60}")
    df = pd.DataFrame(results)
    print(df[["model", "rmse_q95", "rmse_q99", "gate_mean", "alpha_ps"]]
          .to_string(index=False))
    df.to_csv(os.path.join(OUT_BASE, f"summary_{U_MODE}.csv"), index=False)